# UNIDAD I: Introducción a algoritmos y estructuras de datos
## 🐍 Capítulo 1 DSAP: Python Primer — Iteradores, Generadores y Comprehensions
### Programación III - Lic. en Sistemas

![Calculadora](images/pexels-alesiakozik-7289233.jpg)

[Foto de Alesia  Kozik](https://www.pexels.com/es-es/foto/hombre-fuerza-fitnes-fitness-7289233/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap01/03_Iteradores_Generadores_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, el estudiante será capaz de:

1. **Explicar el protocolo iterador** de Python: `__iter__` y `__next__`; diferencia entre iterable e iterador
2. **Implementar clases iteradoras propias** para estructuras de datos
3. **Usar `yield`** para crear generadores que producen valores de forma perezosa
4. **Escribir comprehensions** (list, dict, set) y expresiones generadoras de forma idiomática
5. **Aplicar funciones integradas** (`enumerate`, `zip`, `map`, `filter`, `sorted`, `reversed`) sobre iterables
6. **Importar módulos** con `import` y `from ... import`

---

## 📖 Contenidos

### 🔷 Sección 1: Iteradores y Generadores
1. Protocolo iterador: `__iter__`, `__next__`, `StopIteration`
2. Iterables vs iteradores
3. Implementar un iterador de rango personalizado
4. Generadores con `yield`: evaluación perezosa
5. `yield from` y generadores anidados

### 🔷 Sección 2: Comprehensions y Herramientas Funcionales
1. List comprehensions: `[expr for x in it if cond]`
2. Dict comprehensions: `{k: v for k, v in ...}`
3. Set comprehensions y expresiones generadoras
4. `enumerate`, `zip`, `map`, `filter`, `sorted`, `reversed`, `any`, `all`

### 🔷 Sección 3: Módulos y el Sistema de Importación
1. `import módulo` vs `from módulo import nombre`
2. Módulos estándar útiles: `itertools`, `functools`, `collections`
3. `itertools`: `chain`, `islice`, `count`, `cycle`, `product`
4. `collections`: `Counter`, `deque`, `defaultdict`

---

## 🔧 Configuración del Entorno

In [ ]:
import sys
import itertools
import functools
import collections
import unittest
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)

print(f"Python {sys.version}")
print("Entorno configurado ✅")

---

# 🔷 Sección 1: Iteradores y Generadores

## 🎯 Introducción

> *"Python's for-loop construct is defined to work with any object that is **iterable**. The mechanism by which Python supports iteration is based on the iterator design pattern."*  
> — Goodrich, Tamassia & Goldwasser, §1.8

**Analogía:** Un iterable es como un **libro**; el iterador es el **marcapáginas** — el libro puede tener múltiples marcapáginas independientes, cada uno recuerda en qué página está.

## 📊 Protocolo iterador

| Concepto | Descripción | Método clave |
|----------|-------------|--------------|
| **Iterable** | Objeto que puede producir un iterador | `__iter__()` → iterador |
| **Iterador** | Objeto que produce valores uno a uno | `__next__()` → siguiente valor |
| **`StopIteration`** | Excepción que señala que se agotaron los valores | lanzada por `__next__` |

```
for x in iterable:           # equivale a:
    ...                      it = iter(iterable)      # llama __iter__
                             while True:
                                 try:
                                     x = next(it)    # llama __next__
                                 except StopIteration:
                                     break
```

In [ ]:
# ── Protocolo iterador desde cero ────────────────────────────────────────────
class RangoIterador:
    """
    Iterador personalizado que recorre [inicio, fin) de a pasos.
    Implementa el protocolo iterador de Python (§1.8).
    """
    def __init__(self, inicio: int, fin: int, paso: int = 1):
        if paso == 0:
            raise ValueError("paso no puede ser cero")
        self._inicio = inicio
        self._fin = fin
        self._paso = paso
        self._actual = inicio   # estado del iterador

    def __iter__(self):
        """Un iterador es su propio iterable."""
        return self

    def __next__(self):
        if (self._paso > 0 and self._actual >= self._fin) or \
           (self._paso < 0 and self._actual <= self._fin):
            raise StopIteration
        valor = self._actual
        self._actual += self._paso
        return valor

    def __len__(self):
        """Cuántos elementos produce (sin consumir el iterador)."""
        return max(0, (self._fin - self._inicio + self._paso - 1) // self._paso)


print("=== RangoIterador demo ===")
rango = RangoIterador(0, 10, 2)
print("Elementos:", list(rango))  # convierte todos a lista

# Múltiples iteradores independientes sobre el MISMO objeto
print("\nDos iteradores independientes:")
r1 = RangoIterador(1, 5)
r2 = RangoIterador(1, 5)
print(f"  r1: {next(r1)}, {next(r1)}")   # 1, 2
print(f"  r2: {next(r2)}")                 # 1 — independiente
print(f"  r1: {next(r1)}")                 # 3

# Protocolo explícito: iter() y next()
print("\n--- Protocolo manual ---")
it = iter([10, 20, 30])
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration → iterador agotado")

### ⚡ Generadores con `yield`

Los **generadores** son funciones que producen valores de forma **perezosa** (lazy): no calculan todos los resultados a la vez, sino uno a la vez cuando se solicita.

**Ventaja clave para DS&A**: pueden producir secuencias infinitas o muy largas sin ocupar toda la memoria.

---

# 🔷 Sección 2: Comprehensions y Herramientas Funcionales

## 🎯 Introducción

Las **comprehensions** son una forma concisa y legible de construir colecciones a partir de iterables.  
Las **funciones integradas** sobre iterables (`map`, `filter`, `zip`, etc.) permiten escribir código funcional eficiente.

## 📊 Comparativa

| Construcción | Sintaxis | Produce |
|-------------|---------|---------|
| List comprehension | `[f(x) for x in it if cond]` | `list` |
| Dict comprehension | `{k: v for k, v in it}` | `dict` |
| Set comprehension | `{f(x) for x in it}` | `set` |
| Generator expression | `(f(x) for x in it)` | generador perezoso |

In [ ]:
# ── Generadores con yield ─────────────────────────────────────────────────────
def fibonacci_gen():
    """Generador infinito de Fibonacci."""
    a, b = 0, 1
    while True:
        yield a          # pausa y produce a
        a, b = b, a + b

def primeros_n(gen, n):
    """Toma los primeros n valores de un generador."""
    return [next(gen) for _ in range(n)]

gen = fibonacci_gen()
print("Fibonacci (primeros 12):", primeros_n(gen, 12))

# yield from: delegar en otro generador
def pares(hasta):
    yield from range(0, hasta, 2)   # equivale a: for v in range(...): yield v

def impares(hasta):
    yield from range(1, hasta, 2)

def entrelazar(hasta):
    """Entrelaza pares e impares."""
    for p, i in zip(pares(hasta), impares(hasta)):
        yield p
        yield i

print("\nEntrelazados hasta 10:", list(entrelazar(10)))

# ── Uso de memoria: lista vs generador ───────────────────────────────────────
import sys

def suma_cuadrados_lista(n):
    cuadrados = [i*i for i in range(n)]   # crea toda la lista en memoria
    return sum(cuadrados)

def suma_cuadrados_gen(n):
    gen = (i*i for i in range(n))         # generador: un valor a la vez
    return sum(gen)

n = 1_000_000
lista = [i*i for i in range(n)]
gen   = (i*i for i in range(n))
print(f"\nMemoria lista ({n} elementos): {sys.getsizeof(lista):>10,} bytes")
print(f"Memoria generador:             {sys.getsizeof(gen):>10,} bytes")

### 📈 Análisis visual: comprehension vs loop vs map

In [ ]:
# ── Comprehensions ────────────────────────────────────────────────────────────
print("=== List Comprehension ===")
# Cuadrados de pares entre 0 y 20
cuadrados_pares = [x**2 for x in range(21) if x % 2 == 0]
print("cuadrados_pares:", cuadrados_pares)

# Matriz identidad 4×4
n = 4
identidad = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
print("\nMatriz identidad 4×4:")
for fila in identidad:
    print(" ", fila)

print("\n=== Dict Comprehension ===")
palabras = ["algoritmo", "estructura", "dato", "árbol", "grafo"]
longitudes = {p: len(p) for p in palabras}
print("longitudes:", longitudes)

# Invertir un dict (asumiendo valores únicos)
invertido = {v: k for k, v in longitudes.items()}
print("invertido: ", invertido)

print("\n=== Set Comprehension ===")
texto = "estructuras de datos y algoritmos"
letras_unicas = {c for c in texto if c.isalpha()}
print(f"letras únicas ({len(letras_unicas)}): {sorted(letras_unicas)}")

print("\n=== Generator Expression (suma grande sin lista) ===")
total = sum(x**2 for x in range(1_000_000))
print(f"Suma de cuadrados hasta 999999: {total:,}")

In [ ]:
# ── Funciones integradas sobre iterables ─────────────────────────────────────
print("=== enumerate ===")
frutas = ["manzana", "banana", "cereza"]
for i, fruta in enumerate(frutas, start=1):
    print(f"  {i}. {fruta}")

print("\n=== zip ===")
nombres = ["Ana", "Bruno", "Carla"]
notas   = [85, 92, 78]
for nombre, nota in zip(nombres, notas):
    print(f"  {nombre}: {nota}")

print("\n=== map y filter ===")
numeros = [-3, -1, 0, 2, 4, -5, 7]
positivos   = list(filter(lambda x: x > 0, numeros))
cuadrados   = list(map(lambda x: x**2, positivos))
print(f"  positivos:  {positivos}")
print(f"  cuadrados:  {cuadrados}")

print("\n=== sorted con key ===")
datos = [("Pila",  3), ("Cola", 1), ("Deque", 2), ("Árbol", 5)]
por_prioridad = sorted(datos, key=lambda t: t[1])
print("  por prioridad:", por_prioridad)

print("\n=== any y all ===")
vals = [2, 4, 6, 7, 8]
print(f"  all pares:  {all(v % 2 == 0 for v in vals)}")   # False (7 es impar)
print(f"  any impar:  {any(v % 2 != 0 for v in vals)}")   # True

---

# 🔷 Sección 3: Módulos útiles para DS&A

## 🎯 Introducción

Python viene con una **biblioteca estándar** muy rica. Tres módulos son especialmente útiles en DS&A:

| Módulo | Propósito |
|--------|-----------|
| `itertools` | Iteradores combinatorios y de alta performance |
| `collections` | Contenedores especializados: `deque`, `Counter`, `defaultdict` |
| `functools` | Herramientas funcionales: `reduce`, `lru_cache`, `partial` |

In [ ]:
# ── itertools ─────────────────────────────────────────────────────────────────
print("=== itertools ===\n")
from itertools import chain, islice, count, cycle, product, combinations, permutations

# chain: concatena múltiples iterables
print("chain([1,2], [3,4], [5]):", list(chain([1,2], [3,4], [5])))

# islice: slice de un generador (no soportan indexación)
contador_infinito = count(start=0, step=5)
print("count(0, step=5) primeros 6:", list(islice(contador_infinito, 6)))

# cycle: repetir ciclicamente
turnos = list(islice(cycle(["A", "B", "C"]), 8))
print("cycle(['A','B','C']) × 8:", turnos)

# product: producto cartesiano (útil para búsqueda exhaustiva)
print("product([0,1], repeat=3):", list(product([0,1], repeat=3)))

# combinations y permutations
elementos = [1, 2, 3]
print("combinations([1,2,3], 2):", list(combinations(elementos, 2)))
print("permutations([1,2,3], 2):", list(permutations(elementos, 2)))

# ── collections ───────────────────────────────────────────────────────────────
print("\n=== collections ===\n")
from collections import Counter, deque, defaultdict

# Counter: frecuencias en O(n)
texto = "mississippi"
freq = Counter(texto)
print(f"Counter('{texto}'): {freq}")
print(f"  3 más comunes: {freq.most_common(3)}")

# deque: lista doble extremo O(1) en ambos extremos (cap 6 preview)
cola = deque([1, 2, 3])
cola.appendleft(0)   # O(1)
cola.append(4)       # O(1)
print(f"\ndeque: {cola}")
print(f"  popleft: {cola.popleft()}  → {cola}")

# defaultdict: dict con valor por defecto
grafo = defaultdict(list)   # capítulo 14 preview
grafo["A"].append("B")
grafo["A"].append("C")
grafo["B"].append("D")
print(f"\ngrafo (defaultdict): {dict(grafo)}")

# ── functools ──────────────────────────────────────────────────────────────────
print("\n=== functools ===\n")
from functools import reduce, lru_cache

# reduce: acumular (suma, producto, …)
producto = reduce(lambda a, b: a * b, [1, 2, 3, 4, 5])
print(f"reduce(×, [1..5]): {producto}")

# lru_cache: memoización automática (cap 4 preview)
@lru_cache(maxsize=None)
def fib_memo(n):
    if n < 2: return n
    return fib_memo(n-1) + fib_memo(n-2)

print(f"fib_memo(40): {fib_memo(40)}")
print(f"cache info:   {fib_memo.cache_info()}")

### 🧪 Pruebas con unittest

In [ ]:
class TestIteradoresGeneradores(unittest.TestCase):

    # ── RangoIterador ─────────────────────────────────────────────────────────
    def test_rango_basico(self):
        self.assertEqual(list(RangoIterador(0, 5)), [0, 1, 2, 3, 4])

    def test_rango_paso(self):
        self.assertEqual(list(RangoIterador(0, 10, 2)), [0, 2, 4, 6, 8])

    def test_rango_vacio(self):
        self.assertEqual(list(RangoIterador(5, 5)), [])

    def test_rango_independientes(self):
        r1 = RangoIterador(1, 4)
        r2 = RangoIterador(1, 4)
        self.assertEqual(next(r1), 1)
        self.assertEqual(next(r2), 1)   # r2 no se afectó por r1
        self.assertEqual(next(r1), 2)

    # ── fibonacci_gen ─────────────────────────────────────────────────────────
    def test_fibonacci_primeros(self):
        gen = fibonacci_gen()
        self.assertEqual([next(gen) for _ in range(8)], [0,1,1,2,3,5,8,13])

    # ── Comprehensions ────────────────────────────────────────────────────────
    def test_list_comprehension(self):
        cuad = [x**2 for x in range(5)]
        self.assertEqual(cuad, [0, 1, 4, 9, 16])

    def test_dict_comprehension(self):
        d = {k: k**2 for k in range(4)}
        self.assertEqual(d, {0:0, 1:1, 2:4, 3:9})

    # ── functools ─────────────────────────────────────────────────────────────
    def test_fib_memo(self):
        self.assertEqual(fib_memo(10), 55)
        self.assertEqual(fib_memo(0), 0)
        self.assertEqual(fib_memo(1), 1)

unittest.main(argv=[''], verbosity=2, exit=False)

In [ ]:
# ── Comparación visual: memoria lista vs generador ────────────────────────────
import sys

tamaños = [100, 1_000, 10_000, 100_000, 1_000_000]
mem_lista = [sys.getsizeof([i*i for i in range(n)]) / 1024 for n in tamaños]
# Todos los generadores tienen el mismo tamaño (fijo ~120 bytes)
mem_gen   = [sys.getsizeof(i*i for i in range(n)) / 1024 for n in tamaños]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tamaños, mem_lista, 'b-o', label='list comprehension O(n)', linewidth=2)
ax.plot(tamaños, mem_gen,   'g-s', label='generator expression O(1)', linewidth=2)
ax.set_xlabel('n (número de elementos)')
ax.set_ylabel('Memoria (KB)')
ax.set_title('Memoria: list comprehension vs generator expression')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("→ El generador ocupa memoria constante (~120 bytes) independientemente de n")

In [ ]:
# ── Recorrido perezoso de un árbol binario (preview cap 8) ───────────────────
print("=== Generador para recorrido inorden de árbol binario ===\n")

class NodoBin:
    def __init__(self, val, izq=None, der=None):
        self.val = val
        self.izq = izq
        self.der = der

def inorden(nodo):
    """Generador de recorrido inorden (izq → raíz → der)."""
    if nodo is None:
        return
    yield from inorden(nodo.izq)
    yield nodo.val
    yield from inorden(nodo.der)

# Árbol: [4,2,6,1,3,5,7]
raiz = NodoBin(4,
    NodoBin(2, NodoBin(1), NodoBin(3)),
    NodoBin(6, NodoBin(5), NodoBin(7))
)

resultado = list(inorden(raiz))
print("Recorrido inorden:", resultado)
print("¿Está ordenado?  ", resultado == sorted(resultado))
print("\n→ yield from permite recorridos árbol perezosos en O(1) memoria adicional")

---

## 📊 Resumen del Capítulo 1 Completo

### 🎯 Los tres notebooks en una tabla

| Notebook | Temas clave | Para DS&A |
|----------|-------------|-----------|
| **01** Objetos y Tipos | id/type/value, mutabilidad, list/dict/set, control de flujo | Fundamento de todas las estructuras |
| **02** Funciones y Excepciones | def, *args/**kwargs, LEGB, try/except/finally, excepciones propias | Modularidad y robustez de ADTs |
| **03** Iteradores y Comprehensions | Protocolo iterador, yield, comprehensions, itertools, collections | Recorrido eficiente de estructuras |

### 📈 Complejidades clave del capítulo

| Operación | Estructura | Complejidad |
|-----------|-----------|-------------|
| Acceso por índice | `list`, `str`, `tuple` | O(1) |
| Búsqueda por valor | `list` | O(n) |
| Búsqueda por clave/valor | `dict`, `set` | O(1) promedio |
| `list.append` / `deque.appendleft` | — | O(1) amortizado |
| `list.insert(0, v)` | — | O(n) |
| `Counter(iterable)` | — | O(n) |
| Generator expression | — | O(1) espacio |


---

## 🔍 Autoevaluación Final

1. **¿Cuándo es mejor un `set` que una `list`?** Dar un ejemplo concreto.
2. **¿Qué diferencia hay entre un generador y un iterador?** ¿Todo generador es un iterador?
3. **Escribir una list comprehension** que produzca `[(0,0),(0,1),(1,0),(1,1),(2,0),(2,1)]`  
   (producto cartesiano de `range(3)` × `range(2)`)
4. **¿Por qué `Counter` es más eficiente que hacer manualmente `d[k] = d.get(k,0) + 1`?**
5. **¿Qué ventaja tiene `yield from inorden(nodo.izq)` sobre `for v in inorden(nodo.izq): yield v`?**

---

**🎯 ¡Capítulo 1 completado!**  
**📝 Siguiente**: Ejercicios de clase para consolidar los fundamentos.

---

## 📖 Referencias Bibliográficas

**[DSAP]** Goodrich, M. T., Tamassia, R., & Goldwasser, M. H. (2013). *Data Structures and Algorithms in Python*. John Wiley & Sons.  
- Sección 1.8: Iterators and Generators  
- Sección 1.9: Additional Python Conveniences  
- Sección 1.11: Modules and the Import Statement

**[PyDocs-iter]** Python Software Foundation. (2024). *iterator — Python Glossary*.  
<https://docs.python.org/3/glossary.html#term-iterator>

**[PyDocs-itertools]** Python Software Foundation. (2024). *itertools — Functions creating iterators for efficient looping*.  
<https://docs.python.org/3/library/itertools.html>

**[PyDocs-collections]** Python Software Foundation. (2024). *collections — Container datatypes*.  
<https://docs.python.org/3/library/collections.html>

---

© 2026 Cátedra Programación III — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).